# 301. 删除无效的括号

把字符串看成状态，删除一个括号是一条代价为 1 的边。BFS 找到的第一层合法状态，就是全部最少删除方案。

[原文](https://leetcode.cn/problems/remove-invalid-parentheses/solution/301-shan-chu-wu-xiao-de-gua-hao-bfs-zui-hsh84/) · 整理于 2026-10-17。原代码保持原样；补充注释另放单元格。

## 按删除次数分层

固定本层长度；收集本层所有合法串后停止。合法性要求任意前缀余额非负，最终余额为 0。

In [ ]:
# queue：待处理的字符串；visited：入队时去重。
# size 固定本层数量，新增状态留到下一层。
# found 只在处理完整层后检查，避免漏掉同层答案。
# 只删除括号，字母必须保留。

In [ ]:
from collections import deque

class Solution:
    def removeInvalidParentheses(self, s: str) -> list[str]:

        def is_valid(s):
            left_parentheses_cnt = 0

            for c in s:
                if c == "(":
                    left_parentheses_cnt += 1

                elif c == ")":
                    if left_parentheses_cnt > 0:
                        left_parentheses_cnt -= 1
                    else:
                        return False

            return left_parentheses_cnt == 0

        queue = deque([s])
        visited = {s}
        ans = []

        while queue:
            size = len(queue)
            found = False

            for _ in range(size):
                cur = queue.popleft()

                if is_valid(cur):
                    ans.append(cur)
                    found = True
                    continue

                for i, c in enumerate(cur):

                    if c != "(" and c != ")":
                        continue

                    cand = cur[:i] + cur[i + 1:]

                    if cand not in visited:
                        visited.add(cand)
                        queue.append(cand)

            if found:
                break

        return ans

In [ ]:
assert set(Solution().removeInvalidParentheses("()())()")) == {"(())()", "()()()"}
assert set(Solution().removeInvalidParentheses("(a)())()")) == {"(a())()", "(a)()()"}
assert Solution().removeInvalidParentheses(")(") == [""]

## 易错点与开销

- `append` 配 `popleft` 保持 FIFO；`pop(-1)` 会破坏层序。
- 发现合法串后只处理完当前层；继续收集更深层会混入非最优结果。
- 相同括号删除后可能得到同一字符串，要在入队时去重。

设输入长度为 n、括号数为 p，状态数量上界为 O(2^p)。本实现逐状态切片构造候选，时间保守上界 O(n²·2^p)，空间 O(n·2^p)。